<h2>Modelling: Future Price Direction 4 (LSTM)

<h3>Data Preparation

In [7]:
# Load pre split data:
from project_paths import RAW_DATA_PATH
import pandas as pd

from bta_direction_functions import add_direction
print(RAW_DATA_PATH)

raw_data = pd.read_pickle(RAW_DATA_PATH)
nd_data = raw_data.copy()
nd_data.head()

/Users/cameroncochrane/Data Projects/vlXsnPPgKKnWqEJB/data/raw/btcusd_data.pkl


Price,Date,Close,High,Low,Open,Volume
0,2014-09-17,457.334015,468.174011,452.421997,465.864014,21056800
1,2014-09-18,424.440002,456.859985,413.104004,456.859985,34483200
2,2014-09-19,394.795990,427.834991,384.532013,424.102997,37919700
3,2014-09-20,408.903992,423.295990,389.882996,394.673004,36863600
4,2014-09-21,398.821014,412.425995,393.181000,408.084991,26580100


In [8]:
data_20 = add_direction(data=nd_data, shift=20, price="Close")
data_20 = data_20.drop(columns=["Date"])
data_20.head()

Price,Close,High,Low,Open,Volume,Direction_20
0,457.334015,468.174011,452.421997,465.864014,21056800,0
1,424.440002,456.859985,413.104004,456.859985,34483200,0
2,394.795990,427.834991,384.532013,424.102997,37919700,0
3,408.903992,423.295990,389.882996,394.673004,36863600,0
4,398.821014,412.425995,393.181000,408.084991,26580100,0


In [9]:
from bta_direction_functions import add_EMA, add_SMA

Let's add _5 (1 week), _10 (2 week_), _20 (1 month) SMAs and EMAs. Leave out MACD, RSI and ATR.

In [10]:
data_20 = add_SMA(data=data_20,n=5,price="Close")
data_20 = add_SMA(data=data_20,n=10,price="Close")
data_20 = add_SMA(data=data_20,n=20,price="Close")

data_20 = add_EMA(data=data_20,n=5,price="Close")
data_20 = add_EMA(data=data_20,n=10,price="Close")
data_20 = add_EMA(data=data_20,n=20,price="Close")

In [11]:
data_20 = data_20.dropna().copy() # Drop NA rows
data_20.head()

Price,Close,High,Low,Open,Volume,Direction_20,SMA_5,SMA_10,SMA_20,EMA_5,EMA_10,EMA_20
19,330.079010,345.134003,302.559998,320.389008,79011800,1,342.807800,363.676599,389.910399,341.195946,359.626329,383.883180
20,336.187012,339.247009,320.481995,330.584015,49199900,1,335.030804,357.343301,383.853049,339.526301,355.364635,379.340688
21,352.940002,354.364014,327.187988,336.115997,54736300,1,333.716406,354.919202,380.278049,343.997535,354.923793,376.826337
22,365.026001,382.726013,347.687012,352.747986,83641104,0,340.948407,353.875101,378.789549,351.007024,356.760558,375.702495
23,361.562012,375.066986,352.963013,364.687012,43665700,0,349.158807,351.336902,376.422450,354.525353,357.633549,374.355783


- Remove Volume, Open, SMA_5, and Low in the modelling pipeline based on the feature selection from the previous notebook.
- Temporal train-test split.
- Scale numerical features on training split
- Add overlap sequences
- Train an LSTM.

In [13]:
X = data_20.drop(columns=["Direction_20"])
y = data_20["Direction_20"]

In [45]:
split_idx = int(len(X) * 0.8)
val_idx = int(split_idx * 0.8)

X_train, X_val, X_test = X.iloc[:val_idx], X.iloc[val_idx:split_idx], X.iloc[split_idx:]
y_train, y_val, y_test = y.iloc[:val_idx], y.iloc[val_idx:split_idx], y.iloc[split_idx:]

print(f"Training samples: {len(X_train)} | Validation samples: {len(X_val)} | Test samples: {len(X_test)}")

Training samples: 2790 | Validation samples: 698 | Test samples: 872


<h4>Data Preprocessing Pipeline:

In [20]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

In [30]:
# Function to generate lag features in a pipeline:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

class RollingWindowTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, lookback=96):
        self.lookback = lookback

    def fit(self, X, y=None):
        X = np.asarray(X)
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        check_is_fitted(self)

        X = np.asarray(X, dtype=np.float32)

        if X.ndim != 2:
            raise ValueError("X must be 2D")

        if X.shape[1] != self.n_features_in_:
            raise ValueError("Number of features has changed")

        if not 1 <= self.lookback <= len(X):
            raise ValueError("Invalid lookback")

        return np.stack([
            X[i:i + self.lookback]
            for i in range(len(X) - self.lookback + 1)
        ])

In [34]:
lookback = 96

In [35]:
# Processing pipeleine only not a modelling pipeline:
process_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("rolling", RollingWindowTransformer(lookback=lookback))
])

In [46]:
X_train_seq = process_pipeline.fit_transform(X_train)
X_val_seq = process_pipeline.transform(X_val)
X_test_seq = process_pipeline.transform(X_test)

print(X_train_seq.shape)
print(X_val_seq.shape)
print(X_test_seq.shape)

(2695, 96, 11)
(603, 96, 11)
(777, 96, 11)


In [47]:
# The rolling transformation reduces the number of samples by lookback - 1. Consequently, the target must be aligned with the end of each sequence:

y_train_seq = np.asarray(y_train)[lookback - 1:]
y_val_seq = np.asarray(y_val)[lookback - 1:]
y_test_seq = np.asarray(y_test)[lookback - 1:]

print(y_train_seq.shape)
print(y_val_seq.shape)
print(y_test_seq.shape)

(2695,)
(603,)
(777,)


In [48]:
# Check that the pipeline is fitted. If this runs without an error, the pipeline is recognised as fitted:
check_is_fitted(process_pipeline)

print(process_pipeline.named_steps["rolling"].n_features_in_)

11


<h3>LSTM Model Training

In [41]:
import tensorflow as tf
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

<h4>Model Definition:

In [43]:
tf.keras.utils.set_random_seed(13)

model = Sequential([
    Input(shape=(X_train_seq.shape[1], X_train_seq.shape[2])),

    LSTM(
        units=32,
        return_sequences=False
    ),

    Dropout(0.2),

    Dense(
        units=16,
        activation="relu",
        kernel_regularizer=l2(1e-4)
    ),

    Dense(
        units=1,
        activation="sigmoid"
    )
])

In [44]:
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="roc_auc")
    ]
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 32)             │         5,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,177 (24.13 KB)

 Trainable params: 6,177 (24.13 KB)

 Non-trainable params: 0 (0.00 B)

<h4>Model Training:

In [50]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [52]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=20,
        min_delta=1e-4,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=20,
        min_lr=1e-5
    )
]

history = model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=1000,
    batch_size=64,
    shuffle=False,
    callbacks=callbacks
)


Epoch 1/1000
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.6037 - loss: 0.6621 - roc_auc: 0.6176 - val_accuracy: 0.4378 - val_loss: 0.7160 - val_roc_auc: 0.5458 - learning_rate: 2.5000e-04
Epoch 2/1000
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6100 - loss: 0.6626 - roc_auc: 0.6106 - val_accuracy: 0.4295 - val_loss: 0.7176 - val_roc_auc: 0.5383 - learning_rate: 2.5000e-04
Epoch 3/1000
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6141 - loss: 0.6586 - roc_auc: 0.6247 - val_accuracy: 0.4212 - val_loss: 0.7191 - val_roc_auc: 0.5337 - learning_rate: 2.5000e-04
Epoch 4/1000
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6234 - loss: 0.6601 - roc_auc: 0.6181 - val_accuracy: 0.4129 - val_loss: 0.7208 - val_roc_auc: 0.5301 - learning_rate: 2.5000e-04
Epoch 5/1000
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6260 - loss: 0.6572 - roc_auc: 0.6252 - val_accuracy: 0.4096 - val_loss: 0.7225 - val_roc_auc: 0.5270 - learning_rate: 2.5000e-04
Epoch 6/1000
43